# 08 — Normalization Study: Predicting Win Rate (W/G)

## Strategy Shift
Ridge V1 (public MAE 2.992) remains our best submission despite all complexity attempts.
Lower CV MAE has consistently produced WORSE public scores (Huber CV 2.696 → public 3.029).

**This notebook takes a different approach:**
- Predict **W/G** (win rate, ~0.23–0.75) instead of raw W
- Test 5 normalization variants on a lean 25-feature set
- Drop all era/decade indicators — let normalization handle era effects
- Keep the model simple: Ridge with StandardScaler inside CV

## Normalization Variants
| Variant | Description | Features |
|---------|-------------|----------|
| A | Per-game (÷G) | 25 |
| B | Per-game, then run-scaling stats ÷ mlb_rpg | 24 |
| C | Era z-score of per-game stats | 25 |
| D | A + B + C combined | ~73 |
| E | Best above + era-z-scored target | same |

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# --- Load data ---
train = pd.read_csv('data_year_team_franchise.csv')
test  = pd.read_csv('predict_year_team_franchise.csv')
print(f'Train: {train.shape}, Test: {test.shape}')

# --- Target: W/G (win rate) ---
y_wpct = (train['W'] / train['G']).values
G_train = train['G'].values
G_test  = test['G'].values
test_ids = test['ID'].values

print(f'Target W/G: mean={y_wpct.mean():.4f}, std={y_wpct.std():.4f}, range=[{y_wpct.min():.4f}, {y_wpct.max():.4f}]')

Train: (1812, 52), Test: (453, 48)
Target W/G: mean=0.4983, std=0.0819, range=[0.2338, 0.7484]


## Section 2: Feature Engineering

### 25 Core Features
- **21 counting stats ÷ G** (per-game rates)
- **3 rates from CSV:** ERA, FP, mlb_rpg
- **1 non-linear sabermetric:** pyth_pct = R²/(R²+RA²)

### mlb_rpg Normalization Rules (Variant B)
Only stats that scale with the run-scoring environment get divided by mlb_rpg:

| DIVIDE (10) | LEAVE ALONE (11 counting + 4 rates) |
|-------------|-------------------------------------|
| R, RA, ER, H, 2B, HR, BB, HA, HRA, BBA | AB, 3B, SO, SB, CG, SHO, SV, IPouts, SOA, E, DP, ERA, FP, mlb_rpg, pyth_pct |

In [2]:
# --- Combine train+test for consistent feature engineering ---
train['_is_train'] = True
test['_is_train']  = False
all_data = pd.concat([train, test], ignore_index=True)

# --- 21 counting stats ---
COUNTING_STATS = ['R','AB','H','2B','3B','HR','BB','SO','SB',
                  'RA','ER','CG','SHO','SV','IPouts','HA','HRA','BBA','SOA',
                  'E','DP']

# Stats that scale with the run-scoring environment (divide by mlb_rpg)
RPG_SCALE_STATS = ['R','RA','ER','H','2B','HR','BB','HA','HRA','BBA']

# Stats that DON'T scale with run environment (structural, secular trends)
RPG_LEAVE_STATS = ['AB','3B','SO','SB','CG','SHO','SV','IPouts','SOA','E','DP']

# --- Step 1: Per-game normalization (all 21 stats) ---
for stat in COUNTING_STATS:
    all_data[f'{stat}_pg'] = all_data[stat] / all_data['G']

# --- Step 2: Pythagorean win percentage ---
R  = all_data['R'].values.astype(float)
RA = all_data['RA'].values.astype(float)
all_data['pyth_pct'] = R**2 / (R**2 + RA**2)

PG_COLS = [f'{s}_pg' for s in COUNTING_STATS]
RATE_COLS = ['ERA', 'FP', 'mlb_rpg']
PYTH_COLS = ['pyth_pct']

print(f'Per-game features: {len(PG_COLS)}')
print(f'Rate features: {len(RATE_COLS)}')
print(f'Pyth features: {len(PYTH_COLS)}')
print(f'Total base features: {len(PG_COLS) + len(RATE_COLS) + len(PYTH_COLS)}')

Per-game features: 21
Rate features: 3
Pyth features: 1
Total base features: 25


In [3]:
# --- Step 3: mlb_rpg normalization (Variant B) ---
# Only run-scaling stats get divided by mlb_rpg
RPG_NORM_COLS = []
for stat in RPG_SCALE_STATS:
    col = f'{stat}_rpg'
    all_data[col] = all_data[f'{stat}_pg'] / all_data['mlb_rpg']
    RPG_NORM_COLS.append(col)

# Non-run-scaling stats keep their per-game form in Variant B
RPG_LEAVE_PG_COLS = [f'{s}_pg' for s in RPG_LEAVE_STATS]

print(f'mlb_rpg-normalized features: {len(RPG_NORM_COLS)} → {RPG_NORM_COLS}')
print(f'Left alone (per-game): {len(RPG_LEAVE_PG_COLS)} → {RPG_LEAVE_PG_COLS}')

mlb_rpg-normalized features: 10 → ['R_rpg', 'RA_rpg', 'ER_rpg', 'H_rpg', '2B_rpg', 'HR_rpg', 'BB_rpg', 'HA_rpg', 'HRA_rpg', 'BBA_rpg']
Left alone (per-game): 11 → ['AB_pg', '3B_pg', 'SO_pg', 'SB_pg', 'CG_pg', 'SHO_pg', 'SV_pg', 'IPouts_pg', 'SOA_pg', 'E_pg', 'DP_pg']


In [4]:
# --- Step 4: Era z-score normalization (Variant C) ---
# Assign era labels for grouping (then drop era columns from features)
era_cols = [c for c in all_data.columns if c.startswith('era_')]
all_data['era_label'] = all_data[era_cols].idxmax(axis=1)

# Features to era z-score: all 21 per-game counting stats + ERA + FP
# NOT: mlb_rpg (constant within era, std=0) and pyth_pct (era-independent)
EZ_INPUT_COLS = PG_COLS + ['ERA', 'FP']
EZ_OUTPUT_COLS = []

# Compute era means and stds on combined data
eps = 1e-6
era_stats = {}
for col in EZ_INPUT_COLS:
    grouped = all_data.groupby('era_label')[col]
    era_mean = grouped.transform('mean')
    era_std  = grouped.transform('std') + eps
    ez_col = f'{col}_ez'
    all_data[ez_col] = (all_data[col] - era_mean) / era_std
    EZ_OUTPUT_COLS.append(ez_col)
    # Store stats for Variant E target normalization
    era_stats[col] = {'mean': grouped.mean().to_dict(), 'std': grouped.std().to_dict()}

print(f'Era z-scored features: {len(EZ_OUTPUT_COLS)}')

# Also store era stats for W/G (target) for Variant E
all_data['_wpct'] = all_data['W'] / all_data['G']  # NaN for test rows (no W)
train_mask = all_data['_is_train']
wpct_era_stats = all_data.loc[train_mask].groupby('era_label')['_wpct'].agg(['mean','std']).to_dict()
print(f'W/G era stats for Variant E: computed on {train_mask.sum()} train rows')

Era z-scored features: 23
W/G era stats for Variant E: computed on 1812 train rows


## Section 3: Build Feature Matrices for Each Variant

In [5]:
# --- Split back to train/test ---
tr = all_data[all_data['_is_train']].copy()
te = all_data[~all_data['_is_train']].copy()

def build_matrix(df, feature_cols):
    """Extract feature matrix, replace inf/nan with 0."""
    X = df[feature_cols].values.astype(float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X

# --- Variant A: Per-game + rates + pyth_pct ---
VA_COLS = PG_COLS + RATE_COLS + PYTH_COLS
X_train_A = build_matrix(tr, VA_COLS)
X_test_A  = build_matrix(te, VA_COLS)

# --- Variant B: mlb_rpg-normalized run stats + per-game others + rates (no mlb_rpg) + pyth_pct ---
VB_COLS = RPG_NORM_COLS + RPG_LEAVE_PG_COLS + ['ERA', 'FP'] + PYTH_COLS
X_train_B = build_matrix(tr, VB_COLS)
X_test_B  = build_matrix(te, VB_COLS)

# --- Variant C: Era z-scored counting stats + ERA_ez + FP_ez + mlb_rpg(no) + pyth_pct ---
# mlb_rpg dropped (redundant with z-scoring), pyth_pct kept as-is
VC_COLS = EZ_OUTPUT_COLS + PYTH_COLS
X_train_C = build_matrix(tr, VC_COLS)
X_test_C  = build_matrix(te, VC_COLS)

# --- Variant D: Kitchen sink (A + B + C combined, deduplicated) ---
VD_COLS = list(dict.fromkeys(VA_COLS + VB_COLS + VC_COLS))  # preserve order, dedup
X_train_D = build_matrix(tr, VD_COLS)
X_test_D  = build_matrix(te, VD_COLS)

variants = {
    'A_pergame':    (X_train_A, X_test_A, VA_COLS),
    'B_mlbrpg':     (X_train_B, X_test_B, VB_COLS),
    'C_era_zscore': (X_train_C, X_test_C, VC_COLS),
    'D_combined':   (X_train_D, X_test_D, VD_COLS),
}

for name, (Xtr, Xte, cols) in variants.items():
    print(f'{name}: train={Xtr.shape}, test={Xte.shape}, features={len(cols)}')

A_pergame: train=(1812, 25), test=(453, 25), features=25
B_mlbrpg: train=(1812, 24), test=(453, 24), features=24
C_era_zscore: train=(1812, 24), test=(453, 24), features=24
D_combined: train=(1812, 58), test=(453, 58), features=58


## Section 4: Model Evaluation

10-fold CV with Ridge. StandardScaler fitted inside each fold.
Predict W/G, convert back to W (× G, round, clip), compute MAE on W for Kaggle comparison.

In [6]:
ALPHAS = np.logspace(-3, 4, 300)
KF = KFold(n_splits=10, shuffle=True, random_state=42)

def evaluate_variant(X_train, X_test, y_wpct, G_train, G_test):
    """
    Ridge CV predicting W/G. Returns MAE on W scale + test predictions.
    """
    oof = np.zeros(len(y_wpct))
    alphas_chosen = []
    
    for tr_idx, va_idx in KF.split(X_train):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr = y_wpct[tr_idx]
        
        # Scaler INSIDE fold
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_va_s = sc.transform(X_va)
        
        ridge = RidgeCV(alphas=ALPHAS, cv=5)
        ridge.fit(X_tr_s, y_tr)
        oof[va_idx] = ridge.predict(X_va_s)
        alphas_chosen.append(ridge.alpha_)
    
    # Convert OOF W/G predictions back to W
    oof_W = oof * G_train
    actual_W = y_wpct * G_train
    cv_mae = mean_absolute_error(actual_W, oof_W)
    
    # Final model on all training data
    sc_final = StandardScaler()
    X_all_s = sc_final.fit_transform(X_train)
    ridge_final = RidgeCV(alphas=ALPHAS, cv=10)
    ridge_final.fit(X_all_s, y_wpct)
    
    # Test predictions: W/G → W
    X_test_s = sc_final.transform(X_test)
    test_wpct = ridge_final.predict(X_test_s)
    test_W = np.round(test_wpct * G_test).astype(int)
    test_W = np.clip(test_W, 36, 120)
    
    return cv_mae, np.median(alphas_chosen), test_W, oof

print('Evaluation function ready.')

Evaluation function ready.


In [7]:
# --- Run Variants A-D ---
results = {}
for name, (Xtr, Xte, cols) in variants.items():
    mae, alpha, preds, oof = evaluate_variant(Xtr, Xte, y_wpct, G_train, G_test)
    results[name] = {'mae': mae, 'alpha': alpha, 'preds': preds, 'oof': oof, 'n_feat': len(cols)}
    print(f'{name:20s}: CV MAE = {mae:.5f}  (alpha={alpha:.2f}, features={len(cols)})')

A_pergame           : CV MAE = 2.78694  (alpha=1.75, features=25)


B_mlbrpg            : CV MAE = 2.83136  (alpha=2.50, features=24)


C_era_zscore        : CV MAE = 2.80385  (alpha=0.10, features=24)


D_combined          : CV MAE = 2.73252  (alpha=4.25, features=58)


In [8]:
# --- Variant E: Best variant above + era-z-scored target ---
best_name = min(results, key=lambda k: results[k]['mae'])
best_Xtr, best_Xte, best_cols = variants[best_name]
print(f'Best variant so far: {best_name} (CV MAE = {results[best_name]["mae"]:.5f})')
print(f'Running Variant E with era-z-scored target on top of {best_name}...')

# Era z-score the target W/G using train-only era stats
era_mean_wpct = wpct_era_stats['mean']  # dict: era_label → mean
era_std_wpct  = wpct_era_stats['std']   # dict: era_label → std

train_era_labels = tr['era_label'].values
test_era_labels  = te['era_label'].values

# Transform target
y_wpct_ez = np.array([
    (wp - era_mean_wpct[era]) / (era_std_wpct[era] + 1e-6)
    for wp, era in zip(y_wpct, train_era_labels)
])

# Evaluate with z-scored target
oof_ez = np.zeros(len(y_wpct_ez))
alphas_ez = []

for tr_idx, va_idx in KF.split(best_Xtr):
    X_tr, X_va = best_Xtr[tr_idx], best_Xtr[va_idx]
    y_tr = y_wpct_ez[tr_idx]
    
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    
    ridge = RidgeCV(alphas=ALPHAS, cv=5)
    ridge.fit(X_tr_s, y_tr)
    oof_ez[va_idx] = ridge.predict(X_va_s)
    alphas_ez.append(ridge.alpha_)

# Inverse transform OOF: z-scored → W/G → W
oof_wpct_from_ez = np.array([
    pred_z * (era_std_wpct[era] + 1e-6) + era_mean_wpct[era]
    for pred_z, era in zip(oof_ez, train_era_labels)
])
oof_W_ez = oof_wpct_from_ez * G_train
actual_W = y_wpct * G_train
mae_ez = mean_absolute_error(actual_W, oof_W_ez)

# Final model for test predictions
sc_final = StandardScaler()
X_all_s = sc_final.fit_transform(best_Xtr)
ridge_final = RidgeCV(alphas=ALPHAS, cv=10)
ridge_final.fit(X_all_s, y_wpct_ez)

X_test_s = sc_final.transform(best_Xte)
test_ez_pred = ridge_final.predict(X_test_s)

# Inverse transform test: z-scored → W/G → W
test_wpct_from_ez = np.array([
    pred_z * (era_std_wpct[era] + 1e-6) + era_mean_wpct[era]
    for pred_z, era in zip(test_ez_pred, test_era_labels)
])
test_W_ez = np.round(test_wpct_from_ez * G_test).astype(int)
test_W_ez = np.clip(test_W_ez, 36, 120)

results['E_ez_target'] = {
    'mae': mae_ez, 'alpha': np.median(alphas_ez),
    'preds': test_W_ez, 'oof': oof_wpct_from_ez,
    'n_feat': len(best_cols)
}
print(f'E_ez_target:          CV MAE = {mae_ez:.5f}  (alpha={np.median(alphas_ez):.2f}, features={len(best_cols)})')

Best variant so far: D_combined (CV MAE = 2.73252)
Running Variant E with era-z-scored target on top of D_combined...


E_ez_target:          CV MAE = 2.81698  (alpha=1.49, features=58)


## Section 5: Results Comparison

In [9]:
# --- Results table ---
print('=' * 75)
print(f'{"Variant":20s} {"Features":>8} {"Alpha":>8} {"CV MAE(W)":>12} {"vs Ridge V1":>12}')
print('=' * 75)

ridge_v1_mae = 2.720  # Reference from notebook 01
best_overall = None
best_mae = 999

for name in ['A_pergame', 'B_mlbrpg', 'C_era_zscore', 'D_combined', 'E_ez_target']:
    r = results[name]
    diff = r['mae'] - ridge_v1_mae
    marker = ' *** BEST' if r['mae'] < best_mae else ''
    if r['mae'] < best_mae:
        best_mae = r['mae']
        best_overall = name
    print(f'{name:20s} {r["n_feat"]:>8} {r["alpha"]:>8.2f} {r["mae"]:>12.5f} {diff:>+12.5f}{marker}')

print('=' * 75)
print(f'Reference: Ridge V1 (raw W target, 121 features): CV MAE = {ridge_v1_mae:.3f}, Public = 2.992')
print(f'Reference: V6-A (raw W target, 140 features):     CV MAE = 2.709')
print(f'Reference: V7-Huber (raw W target, 140 features): CV MAE = 2.696, Public = 3.029')
print('=' * 75)
print(f'\nBest variant: {best_overall} (CV MAE = {best_mae:.5f})')

Variant              Features    Alpha    CV MAE(W)  vs Ridge V1
A_pergame                  25     1.75      2.78694     +0.06694 *** BEST
B_mlbrpg                   24     2.50      2.83136     +0.11136
C_era_zscore               24     0.10      2.80385     +0.08385
D_combined                 58     4.25      2.73252     +0.01252 *** BEST
E_ez_target                58     1.49      2.81698     +0.09698
Reference: Ridge V1 (raw W target, 121 features): CV MAE = 2.720, Public = 2.992
Reference: V6-A (raw W target, 140 features):     CV MAE = 2.709
Reference: V7-Huber (raw W target, 140 features): CV MAE = 2.696, Public = 3.029

Best variant: D_combined (CV MAE = 2.73252)


## Section 6: Generate Submission

In [10]:
# --- Save best variant submission ---
best_preds = results[best_overall]['preds']
sub = pd.DataFrame({'ID': test_ids, 'W': best_preds})
sub = sub.sort_values('ID')

fname = f'submissions/v9_{best_overall}.csv'
sub.to_csv(fname, index=False)
print(f'Saved: {fname}')
print(f'Predictions: mean={best_preds.mean():.1f}, range=[{best_preds.min()}, {best_preds.max()}]')
print(sub.head(10))

# --- Also save all variants for comparison submission ---
for name, r in results.items():
    s = pd.DataFrame({'ID': test_ids, 'W': r['preds']}).sort_values('ID')
    s.to_csv(f'submissions/v9_{name}.csv', index=False)
print(f'\nAll {len(results)} variant submissions saved.')

Saved: submissions/v9_D_combined.csv
Predictions: mean=78.9, range=[45, 109]
     ID   W
449   9  68
416  14  75
199  19  92
374  24  85
243  26  77
123  30  82
92   45  96
178  49  87
367  50  82
130  52  82

All 5 variant submissions saved.


## Conclusion

This notebook tested 5 normalization strategies on a lean 25-feature set, predicting W/G instead of raw W.

**Key findings:**
- Which normalization generalizes best (lowest CV MAE on W)?
- Does predicting W/G instead of W change the results?
- Does era-normalizing the target help or hurt?

Next step: submit the best variant to Kaggle to compare public MAE with Ridge V1 (2.992).